# 3 - Cotizaciones 
Se obtiene el precio del dolar y los tickets por el día que se realizó la transación.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *

import pandas as pd
import requests

try:
    import yfinance as yf
except ImportError:
    %pip install yfinance
    import yfinance as yf

In [0]:
START_DATE = "2026-01-01"
END_DATE = "2026-04-01"

df_ops = spark.table("bronze_byma.operaciones_raw")
df_tipo_cambio = spark.table("silver_byma.tipo_cambio")

In [0]:
df_base = (
    df_ops
    .select(
        F.col("simbolo_titulo").alias("simbolo"),
        F.col("fecha_particion").cast("date").alias("fecha")
    )
    .distinct()
)

pdf = df_base.select("simbolo", "fecha").toPandas()
pdf["fecha"] = pd.to_datetime(pdf["fecha"])


In [0]:
pdf = df_base.select("simbolo", "fecha").toPandas()
pdf["fecha"] = pd.to_datetime(pdf["fecha"])

In [0]:
results = []

for symbol, df_group in pdf.groupby("simbolo"):
    
    min_date = df_group["fecha"].min()
    max_date = df_group["fecha"].max()

    try:
        data = yf.download(
            tickers=[symbol, f"{symbol}.BA"],
            start=min_date,
            end=max_date + pd.Timedelta(days=1),
            group_by="ticker",
            progress=False
        )

        for _, row in df_group.iterrows():
            date = row["fecha"]
            price = None

            # intento 1: original
            if symbol in data and not data[symbol].empty:
                df_tmp = data[symbol]
                if date in df_tmp.index:
                    price = df_tmp.loc[date]["Close"]

            # intento 2: .BA
            if price is None and f"{symbol}.BA" in data and not data[f"{symbol}.BA"].empty:
                df_tmp = data[f"{symbol}.BA"]
                if date in df_tmp.index:
                    price = df_tmp.loc[date]["Close"]

            results.append({
                "simbolo": symbol,
                "fecha": date,
                "precio": price,
                "fuente": "yfinance" if price is not None else None
            })

    except:
        for _, row in df_group.iterrows():
            results.append({
                "simbolo": symbol,
                "fecha": row["fecha"],
                "precio": None,
                "fuente": None
            })

df_prices = pd.DataFrame(results)

In [0]:
df_final = pdf.merge(df_prices, on=["simbolo", "fecha"], how="left")

In [0]:
def get_data912_history(symbol):
    endpoints = [
        f"https://data912.com/historical/stocks/{symbol}",
        f"https://data912.com/historical/cedears/{symbol}",
        f"https://data912.com/historical/bonds/{symbol}",
    ]

    for url in endpoints:
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            data = r.json()

            if data and len(data) > 0:
                df = pd.DataFrame(data)
                df["date"] = pd.to_datetime(df["date"]).dt.date
                return df[["date", "c"]]

        except:
            continue

    return None

In [0]:
symbols_missing = df_final[df_final["precio"].isna()]["simbolo"].unique()

data912_hist = {}

for symbol in symbols_missing:
    df_hist = get_data912_history(symbol)
    
    if df_hist is not None:
        data912_hist[symbol] = df_hist

In [0]:
df_final["fecha"] = pd.to_datetime(df_final["fecha"]).dt.date

def completar_data912(row):
    if pd.isna(row["precio"]):
        symbol = row["simbolo"]
        date = row["fecha"]

        df_hist = data912_hist.get(symbol)

        if df_hist is not None:
            match = df_hist[df_hist["date"] == date]

            if not match.empty:
                return match["c"].iloc[0], "data912_hist"

    return row["precio"], row["fuente"]

In [0]:
df_final[["precio", "fuente"]] = df_final.apply(
    lambda row: pd.Series(completar_data912(row)),
    axis=1
)

In [0]:
df_final = spark.createDataFrame(df_final)

In [0]:
url = "https://api.argentinadatos.com/v1/cotizaciones/dolares"

response = requests.get(url)
data = response.json()

df_dolar = pd.DataFrame(data)

# filtrar CCL
df_dolar = df_dolar[
    df_dolar["casa"].str.lower() == "contadoconliqui"
].copy()

# normalizar fecha
df_dolar["fecha"] = pd.to_datetime(df_dolar["fecha"]).dt.date

# nos quedamos con lo necesario
df_dolar = df_dolar[["fecha", "venta"]].rename(
    columns={"venta": "dolar_ccl"}
)

df_dolar_spark = spark.createDataFrame(df_dolar)

df_dolar_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_byma.fecha_dolar")

In [0]:
# 1) crear df dolar spark con nombre único
df_dolar_spark = spark.createDataFrame(df_dolar) \
    .withColumnRenamed("dolar_ccl", "dolar_ccl_new")

# 2) si df_final ya trae columnas viejas del join anterior, eliminarlas
cols_to_drop = [c for c in ["dolar_ccl", "dolar_ccl_new", "precio_usd"] if c in df_final.columns]
df_final = df_final.drop(*cols_to_drop)

# 3) join limpio
df_final = df_final.join(
    df_dolar_spark,
    on="fecha",
    how="left"
)

# 4) conversión correcta usando la columna renombrada
df_final = df_final.withColumn(
    "precio_usd",
    when(
        (col("fuente") == "data912_hist") &
        col("precio").isNotNull() &
        col("dolar_ccl_new").isNotNull(),
        col("precio") / col("dolar_ccl_new")
    ).otherwise(col("precio"))
)

In [0]:
df_final = df_final.filter(col("precio_usd").isNotNull())

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_byma.precios_historicos")